# syft-bg Integration Tests

Reusable notebook that tests `syft_bg` service management:

1. **Config & Status basics** — init, inspect config, check status
2. **Misconfigured `email_approve`** — missing `gcp_project_id` → graceful error
3. **Start / Stop / Restart** — verify PID lifecycle
4. **Auto-approve job via API** — `auto_approve_job()`, follow-up auto-approval, list/remove rules

### Prerequisites
- `.env` file with `DO_EMAIL`, `DS_EMAIL`, `GCP_PROJECT_ID`, `APP_CREDENTIALS`, `TOKEN_DS`
- OAuth credential files (see Setup cells for first-time token generation)

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup: Environment & Credentials

In [ ]:
import os
from pathlib import Path

for line in Path(".env").read_text().splitlines():
    if line.strip() and not line.startswith("#"):
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip())

DO_EMAIL = os.environ["DO_EMAIL"]
DS_EMAIL = os.environ["DS_EMAIL"]
GCP_PROJECT_ID = os.environ["GCP_PROJECT_ID"]

APP_CREDENTIALS = Path(os.environ["APP_CREDENTIALS"])
TOKEN_DS = Path(os.environ["TOKEN_DS"])
DO_CREDENTIALS = APP_CREDENTIALS / "test_app_credentials.json"
DO_TOKEN = Path(os.environ["TOKEN_DO_WITH_SCOPES"])

assert APP_CREDENTIALS.exists(), f"App credentials missing: {APP_CREDENTIALS}"
assert DO_CREDENTIALS.exists(), f"DO credentials missing: {DO_CREDENTIALS}"

print(f"DO: {DO_EMAIL}")
print(f"DS: {DS_EMAIL}")

### First-time token generation

Uncomment and run these cells once to generate OAuth tokens. Each opens a browser for consent.

In [ ]:
import syft_client as sc

# --- DS token (Drive scopes only) ---
# sc.credentials_to_token(
#     credentials_path=str(APP_CREDENTIALS),
#     output_path=str(TOKEN_DS),
# )
# print(f"DS token saved to: {TOKEN_DS}")

In [ ]:
# --- DO token (Drive + Gmail + PubSub scopes) ---
# sc.credentials_to_token(
#     credentials_path=str(DO_CREDENTIALS),
#     output_path=str(DO_TOKEN),
#     do_scopes=True,
# )
# print(f"DO token saved to: {DO_TOKEN}")

In [ ]:
assert TOKEN_DS.exists(), f"DS token missing: {TOKEN_DS} — uncomment generation cell above"
assert DO_TOKEN.exists(), f"DO token missing: {DO_TOKEN} — uncomment generation cell above"
print("All tokens ready")

## Setup: Login & Peers

In [ ]:
DO_EMAIL

In [ ]:
import syft_client as sc

do_client = sc.login_do(email=DO_EMAIL, token_path=str(DO_TOKEN))
ds_client = sc.login_ds(email=DS_EMAIL, token_path=str(TOKEN_DS))

In [ ]:
# sc.delete_syftbox(email=DO_EMAIL, token_path=DO_TOKEN)
# sc.delete_syftbox(email=DS_EMAIL, token_path=TOKEN_DS)

In [ ]:
ds_client.add_peer(DO_EMAIL)
do_client.load_peers()
do_client.approve_peer_request(DS_EMAIL, peer_must_exist=False)
print("Peer relationship established")

## Setup: Test Dataset

In [ ]:
try:
    do_client.create_dataset(
        name="testdata",
        mock_path="data/mock.txt",
        private_path="data/private.txt",
        users=[DS_EMAIL],
    )
except FileExistsError:
    print("Dataset 'testdata' already exists, skipping creation")

In [ ]:
do_client.sync()

---
## Test 1: Config & Status Basics

In [ ]:
import syft_bg

syft_bg.reset()

In [ ]:
syft_bg.init(
    do_email=DO_EMAIL,
    token_path=str(DO_TOKEN),
    settings={
        "email_approve": {"gcp_project_id": GCP_PROJECT_ID},
    },
)

In [ ]:
# Inspect config — should show do_email, token paths, and gcp_project_id
# syft_bg.config

In [ ]:
# syft_bg.reset()

In [ ]:
syft_bg.ensure_running(services=["sync", "notify", "approve", "email_approve"], restart=True)

In [ ]:
# Status before starting any services — all should be stopped
syft_bg.status

In [ ]:
syft_bg.logs("sync")

In [ ]:
for i in range(5):
    do_client.sync()

In [ ]:
do_client.sync()

# Submit job

In [ ]:
import uuid

In [ ]:
%%writefile /tmp/job.py
print("A")

In [ ]:

JOB_NAME = f"api_auto_{uuid.uuid4().hex[:8]}"

ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path="/tmp/job.py",
    job_name="test2",
)
print(f"Submitted: {JOB_NAME}")

In [ ]:
do_client.jobs

In [ ]:
# Clean slate with proper config
syft_bg.reset()
syft_bg.init(
    do_email=DO_EMAIL,
    token_path=str(DO_TOKEN),
    settings={
        "email_approve": {"gcp_project_id": GCP_PROJECT_ID},
    },
)

In [ ]:
# Start sync, notify, approve (skip email_approve — it needs Pub/Sub infra)
# Start sync first so it creates the cache directory before approve needs it
SERVICES = ["sync", "notify", "approve"]

syft_bg.ensure_running(services=["sync"], restart=True)
time.sleep(10)  # let sync create .cache dir

syft_bg.ensure_running(services=["notify", "approve"], restart=True)
time.sleep(5)

status = syft_bg.status
print(status)

In [ ]:
# Record initial PIDs
initial_pids = {}
for svc in SERVICES:
    info = status.service_infos[svc]
    assert info.status in (ServiceStatus.RUNNING, ServiceStatus.STARTING), (
        f"{svc} should be running, got {info.status}"
    )
    initial_pids[svc] = info.pid
    print(f"{svc}: PID {info.pid} ({info.status.value})")

print(f"\nAll {len(SERVICES)} services running")

In [ ]:
# Restart the approve service — PID should change
syft_bg.restart("approve")
time.sleep(3)

new_status = syft_bg.status
new_approve_info = new_status.service_infos["approve"]
assert new_approve_info.pid != initial_pids["approve"], (
    f"approve PID should have changed: was {initial_pids['approve']}, still {new_approve_info.pid}"
)
print(f"approve restarted: PID {initial_pids['approve']} -> {new_approve_info.pid}")

In [ ]:
# Stop approve
syft_bg.stop("approve")
time.sleep(1)

stopped_status = syft_bg.status
assert stopped_status.service_infos["approve"].status == ServiceStatus.STOPPED, (
    f"approve should be stopped, got {stopped_status.service_infos['approve'].status}"
)
print("approve stopped")

# Other services should still be running
for svc in ["sync", "notify"]:
    assert stopped_status.service_infos[svc].status in (ServiceStatus.RUNNING, ServiceStatus.STARTING), (
        f"{svc} should still be running"
    )
print("sync and notify still running")

In [ ]:
# Start approve again
syft_bg.start("approve")
time.sleep(3)

restarted_status = syft_bg.status
assert restarted_status.service_infos["approve"].status in (ServiceStatus.RUNNING, ServiceStatus.STARTING), (
    f"approve should be running again, got {restarted_status.service_infos['approve'].status}"
)
print(f"approve running again: PID {restarted_status.service_infos['approve'].pid}")
print("\nStart/Stop/Restart test passed")